# CNJ — extração completa dos assuntos (notebook independente)

Este arquivo contém todo o código, sem precisar de módulos `.py` externos.

1. Instale as dependências, se necessário, na primeira célula de código.
2. Execute as células na ordem.
3. Se sua rede bloqueia o serviço HTTPS do CNJ, obtenha as planilhas oficiais no navegador e salve em `planilhas_cnj/`; na última célula, use `MODE = "excel"`.
4. Para a coleta online, use `MODE = "soap"`. A opção inicial `"diagnose"` testa a conexão.

Fonte: https://www.cnj.jus.br/sgt/versoes.php?tipo_tabela=A.

A cobertura do modo Excel depende dos arquivos oficiais fornecidos; a coleta SOAP só pode ser considerada completa após terminar com sucesso.


In [1]:
# Execute se estas bibliotecas ainda não estiverem instaladas no ambiente.
%pip install requests==2.32.3 openpyxl==3.1.5 xlrd==2.0.1



  Attempting uninstall: requests

    Found existing installation: requests 2.32.5

   -------------------- ------------------- 1/2 [requests]
    Uninstalling requests-2.32.5:
   -------------------- ------------------- 1/2 [requests]
      Successfully uninstalled requests-2.32.5
   -------------------- ------------------- 1/2 [requests]
   ---------------------------------------- 2/2 [requests]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlex 3.4.2 requires PyYAML==6.0.2, but you have pyyaml 6.0.3 which is incompatible.
databricks-sdk 0.73.0 requires protobuf!=5.26.*,!=5.27.*,!=5.28.*,!=5.29.0,!=5.29.1,!=5.29.2,!=5.29.3,!=5.29.4,!=6.30.0,!=6.30.1,!=6.31.0,<7.0,>=4.25.8, but you have protobuf 7.35.1 which is incompatible.
mlflow 3.6.0 requires pandas<3, but you have pandas 3.0.5 which is incompatible.
mlflow 3.6.0 requires pyarrow<23,>=4.0.0, but you have pyarrow 25.0.1 which is incompatible.
mlflow-skinny 3.6.0 requires packaging<26, but you have packaging 26.3 which is incompatible.
mlflow-skinny 3.6.0 requires protobuf<7,>=3.12.0, but you have protobuf 7.35.1 which is incompatible.
mlflow-tracing 3.6.0 requires packaging<26, but you have packaging 26.3 which is incompatible.
mlflow-tracing 3.6.0 requires protobuf<7,>=3.12.0, but you have prot

In [2]:
# Módulo local de conversão de planilhas oficiais do CNJ.
# É carregado somente em memória; não escreve scripts no disco.
import sys
import types
offline_module = types.ModuleType("offline_excel")
sys.modules["offline_excel"] = offline_module
source = r'''
"""Converte planilhas oficiais de Assuntos do CNJ em um dicionário CSV.

A origem são arquivos baixados manualmente em
https://www.cnj.jus.br/sgt/versoes.php?tipo_tabela=A
Não pressupõe que os layouts sejam idênticos entre todas as versões.
"""
from __future__ import annotations

import csv
import os
import re
import tempfile
import unicodedata
from collections import defaultdict, deque
from pathlib import Path
from html.parser import HTMLParser
import xml.etree.ElementTree as ET
from typing import Iterator, Iterable

FORMATS = {".xlsx", ".xls", ".csv"}
CODE_HEADERS = {"CODIGO", "CODIGO ASSUNTO", "CODIGO DO ASSUNTO", "COD ASSUNTO", "SEQ ELEMENTO", "SEQUENCIAL", "CODIGO DO ITEM"}
DESCRIPTION_HEADERS = {"DESCRICAO", "DESCRICAO DO ASSUNTO", "ASSUNTO", "NOME", "NOME DO ASSUNTO", "TITULO", "DESCRICAO ASSUNTO"}
PARENT_HEADERS = {"CODIGO PAI", "CODIGO DO PAI", "CODIGO ASSUNTO PAI", "SEQ ELEMENTO PAI", "PAI"}
LEVEL_HEADERS = {"NIVEL", "NIVEL DO ASSUNTO"}


def normalize_header(value: object) -> str:
    """Padroniza cabeçalhos para identificar colunas mesmo com acentos."""
    original = unicodedata.normalize("NFKD", str(value or ""))
    ascii_text = "".join(c for c in original if not unicodedata.combining(c))
    return " ".join(re.sub(r"[^A-Z0-9]+", " ", ascii_text.upper()).split())


def to_code(value: object) -> str:
    """Valida códigos positivos e impede conversões incorretas de texto."""
    if isinstance(value, bool) or value is None:
        return ""
    if isinstance(value, int):
        return str(value) if value > 0 else ""
    if isinstance(value, float):
        return str(int(value)) if value > 0 and value.is_integer() else ""
    raw = str(value).strip()
    return raw if raw.isascii() and raw.isdigit() and int(raw) > 0 else ""



class HtmlTableParser(HTMLParser):
    """Lê planilhas XLS publicadas como HTML, sem carregar todas as linhas."""

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self.table_depth = 0
        self.current_row: list[str] | None = None
        self.current_cell: list[str] | None = None
        self.completed: deque[tuple[str, ...]] = deque()

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        tag = tag.lower()
        if tag == 'table':
            self.table_depth += 1
        elif self.table_depth and tag == 'tr':
            self.current_row = []
        elif self.current_row is not None and tag in ('td', 'th'):
            self.current_cell = []

    def handle_data(self, data: str) -> None:
        if self.current_cell is not None:
            self.current_cell.append(data)

    def handle_endtag(self, tag: str) -> None:
        tag = tag.lower()
        if tag in ('td', 'th') and self.current_cell is not None and self.current_row is not None:
            self.current_row.append(' '.join(''.join(self.current_cell).split()))
            self.current_cell = None
        elif tag == 'tr' and self.current_row is not None:
            self.completed.append(tuple(self.current_row))
            self.current_row = None
        elif tag == 'table' and self.table_depth:
            self.table_depth -= 1


def iter_html_rows(path: Path) -> Iterator[tuple[object, ...]]:
    """Processa arquivos HTML mesmo quando a extensão é .xls."""
    parser = HtmlTableParser()
    with path.open(encoding='utf-8-sig', errors='replace') as stream:
        while chunk := stream.read(128 * 1024):
            parser.feed(chunk)
            while parser.completed:
                yield parser.completed.popleft()
    parser.close()
    while parser.completed:
        yield parser.completed.popleft()


def iter_xml_spreadsheet_rows(path: Path) -> Iterator[tuple[object, ...]]:
    """Lê XML Spreadsheet 2003 quando o CNJ publica XLS nesse formato."""
    for _, element in ET.iterparse(path, events=('end',)):
        if element.tag.split('}')[-1] != 'Row':
            continue
        row: list[str] = []
        for cell in element:
            if cell.tag.split('}')[-1] != 'Cell':
                continue
            index = next((int(value) for key, value in cell.attrib.items()
                          if key.split('}')[-1] == 'Index'), len(row) + 1)
            if index < 1 or index > 1024:
                raise ValueError('Índice de coluna inválido na planilha XML.')
            while len(row) < index - 1:
                row.append('')
            data = next((child for child in cell if child.tag.split('}')[-1] == 'Data'), None)
            row.append(''.join(data.itertext()).strip() if data is not None else '')
        yield tuple(row)
        element.clear()

def iter_tables(file_path: Path) -> Iterator[tuple[str, Iterator[tuple[object, ...]]]]:
    """Lê uma aba por vez, sem copiar os arquivos descompactados para o disco."""
    extension = file_path.suffix.lower()
    if extension == ".xlsx":
        try:
            import openpyxl
        except ImportError as exc:
            raise RuntimeError("Instale openpyxl: python -m pip install openpyxl") from exc
        workbook = openpyxl.load_workbook(file_path, read_only=True, data_only=True)
        try:
            for sheet in workbook:
                yield sheet.title, sheet.iter_rows(values_only=True)
        finally:
            workbook.close()
    elif extension == ".xls":
        # Algumas exportações antigas do Excel são tabelas HTML ou XML,
        # apesar de usarem a extensão .xls. Evita exigir xlrd nesses casos.
        with file_path.open('rb') as handle:
            marker = handle.read(256).lstrip(b'\xef\xbb\xbf \t\r\n').lower()
        if marker.startswith((b'<html', b'<!doctype html', b'<table')):
            yield file_path.stem, iter_html_rows(file_path)
            return
        if marker.startswith(b'<?xml') or marker.startswith(b'<workbook'):
            yield file_path.stem, iter_xml_spreadsheet_rows(file_path)
            return
        try:
            import xlrd
        except ImportError as exc:
            raise RuntimeError("Instale xlrd: python -m pip install xlrd") from exc
        workbook = xlrd.open_workbook(str(file_path), on_demand=True)
        try:
            for sheet in workbook.sheets():
                yield sheet.name, (tuple(sheet.row_values(i)) for i in range(sheet.nrows))
        finally:
            workbook.release_resources()
    elif extension == ".csv":
        # CSV de apoio: formato normalizado, com as colunas codigo e descricao.
        # Caso use outra codificação/separador, converta o CSV antes da importação.
        with file_path.open(encoding="utf-8-sig", newline="") as stream:
            first_line = stream.readline()
            stream.seek(0)
            delim = ";" if first_line.count(";") >= first_line.count(",") else ","
            yield file_path.stem, (tuple(row) for row in csv.reader(stream, delimiter=delim))
    else:
        raise ValueError(f"Formato não suportado: {file_path.suffix}")


def _find_header(row: tuple[object, ...]) -> tuple[int, int, int | None, int | None] | None:
    """Reconhece somente layouts explícitos, evitando inventar códigos."""
    labels = [normalize_header(cell) for cell in row]
    codes = [i for i, text in enumerate(labels) if text in CODE_HEADERS]
    descriptions = [i for i, text in enumerate(labels) if text in DESCRIPTION_HEADERS]
    if len(codes) != 1 or len(descriptions) != 1 or codes[0] == descriptions[0]:
        return None
    parents = [i for i, text in enumerate(labels) if text in PARENT_HEADERS]
    levels = [i for i, text in enumerate(labels) if text in LEVEL_HEADERS]
    return codes[0], descriptions[0], parents[0] if parents else None, levels[0] if levels else None


def parse_table(rows: Iterable[tuple[object, ...]], source: str) -> list[tuple[str, str, str, str]]:
    """Extrai registros apenas quando cabeçalhos e códigos são inequívocos."""
    header: tuple[int, int, int | None, int | None] | None = None
    found: list[tuple[str, str, str, str]] = []
    for line_number, cells in enumerate(rows, start=1):
        cells = tuple(cells)
        if header is None:
            header = _find_header(cells)
            # Cabeçalho pode ser precedido por títulos e notas no arquivo oficial.
            if header is None and line_number >= 80:
                break
            continue
        idx_code, idx_description, idx_parent, idx_level = header
        if len(cells) <= max(idx_code, idx_description):
            continue
        code = to_code(cells[idx_code])
        description = str(cells[idx_description] or "").strip()
        if not code or not description:
            continue
        parent = to_code(cells[idx_parent]) if idx_parent is not None and len(cells) > idx_parent else ""
        level = to_code(cells[idx_level]) if idx_level is not None and len(cells) > idx_level else ""
        if parent == code:
            raise ValueError(f"Pai igual ao código na origem {source}, linha {line_number}")
        found.append((code, description, parent, level))
    if header is None:
        raise ValueError(f"Cabeçalhos 'código' e 'descrição/assunto' não encontrados em {source}. "
                         "O layout deve ser inspecionado antes da conversão; nenhuma linha foi deduzida.")
    return found


def convert_excel_directory(input_dir: Path, output_csv: Path) -> int:
    """Une planilhas sem registros duplicados e cria o CSV apenas após validar."""
    input_dir = Path(input_dir).expanduser().resolve()
    output_csv = Path(output_csv).expanduser().resolve()
    if not input_dir.is_dir():
        raise ValueError(f"Pasta não encontrada: {input_dir}")
    files = sorted(p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() in FORMATS
                   and p.resolve() != output_csv)
    if not files:
        raise ValueError(f"Nenhuma planilha XLS, XLSX ou CSV encontrada em {input_dir}")

    records: dict[str, tuple[str, str, str]] = {}
    origins: dict[str, set[str]] = defaultdict(set)
    for file_path in files:
        found_any = False
        for sheet_name, rows in iter_tables(file_path):
            source = f"{file_path.name}/{sheet_name}"
            try:
                parsed = parse_table(rows, source)
            except ValueError as exc:
                # Não ignora abas com aparência de tabela de assuntos mas estrutura desconhecida.
                # Abas sem cabeçalho são informadas como ignoradas para auditoria.
                if "Cabeçalhos" in str(exc):
                    print(f"Aviso: aba ignorada sem cabeçalho reconhecido: {source}")
                    continue
                raise
            for code, description, parent, level in parsed:
                old = records.get(code)
                if old is not None and normalize_header(old[0]) != normalize_header(description):
                    raise ValueError(f"Código {code} apresenta descrições conflitantes: {old[0]!r} e {description!r} ({source}).")
                if old is not None and old[1] and parent and old[1] != parent:
                    raise ValueError(f"Código {code} apresenta códigos pai conflitantes ({source}).")
                records[code] = (description, (old[1] if old and old[1] else parent), (old[2] if old and old[2] else level))
                origins[code].add(file_path.name)
                found_any = True
        if not found_any:
            raise ValueError(f"Nenhum assunto identificado em {file_path.name}. "
                             "Verifique se o arquivo é a exportação tabular de Assuntos do CNJ.")

    if not records:
        raise ValueError("Nenhum assunto identificado; saída não será criada.")
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    temp_file = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8-sig", newline="", suffix=".tmp",
                                         dir=output_csv.parent, delete=False) as stream:
            temp_file = Path(stream.name)
            writer = csv.writer(stream, delimiter=";")
            writer.writerow(["codigo", "descricao", "codigo_pai", "nivel", "fontes"])
            for code in sorted(records, key=int):
                description, parent, level = records[code]
                writer.writerow([code, description, parent, level, ", ".join(sorted(origins[code]))])
        os.replace(temp_file, output_csv)
    finally:
        if temp_file:
            temp_file.unlink(missing_ok=True)
    print(f"CSV gerado: {output_csv} | assuntos únicos: {len(records)} | arquivos: {len(files)}")
    print("A abrangência da tabela depende dos arquivos oficiais selecionados; "
          "não é possível atestar completude nacional por esta conversão isolada.")
    return len(records)

'''
exec(compile(source, "offline_excel.py", "exec"), offline_module.__dict__)
print("Conversor de planilhas carregado.")


Conversor de planilhas carregado.


In [3]:
# Módulo de consulta SOAP e diagnóstico de rede.
# É carregado somente em memória; sem executar o modo CLI.
cnj_module = types.ModuleType("cnj_assuntos_extrator")
sys.modules["cnj_assuntos_extrator"] = cnj_module
source = r'''
"""Extrai a árvore de assuntos processuais do CNJ para um CSV único.

Fonte: WebService público do SGT, consultado pela mesma base da página
https://www.cnj.jus.br/sgt/consulta_publica_assuntos.php.

As categorias raiz foram conferidas com a versão exibida em 12/09/2026.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import logging
import os
import tempfile
import threading
import xml.etree.ElementTree as ET
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Protocol


# ================================================================
# 1. CONFIGURAÇÕES ALTERÁVEIS
# ================================================================

WSDL_URL = "https://www.cnj.jus.br/sgt/sgt_ws.php?wsdl"
OUTPUT_CSV = Path("cnj_assuntos_completos.csv")
MAX_WORKERS = 2                  # Reduz a carga para o WebService legado.
NETWORK_TIMEOUT_SECONDS = 35
NETWORK_RETRIES = 2
MIN_EXPECTED_RECORDS = 1000     # Barreira contra exportações aparentemente completas, mas vazias.

# Categorias de primeiro nível exibidas na página de assuntos do CNJ.
# Atualizadas conforme a versão da interface consultada em 12/09/2026.
ROOT_SUBJECTS: dict[int, str] = {
    12775: "DIREITO À EDUCAÇÃO",
    9985: "DIREITO ADMINISTRATIVO E OUTRAS MATÉRIAS DE DIREITO PÚBLICO",
    10110: "DIREITO AMBIENTAL",
    12734: "DIREITO ASSISTENCIAL",
    899: "DIREITO CIVIL",
    9633: "DIREITO DA CRIANÇA E DO ADOLESCENTE",
    12480: "DIREITO DA SAÚDE",
    1156: "DIREITO DO CONSUMIDOR",
    864: "DIREITO DO TRABALHO",
    11428: "DIREITO ELEITORAL",
    10739: "DIREITO ELEITORAL E PROCESSO ELEITORAL DO STF",
    6191: "DIREITO INTERNACIONAL",
    1146: "DIREITO MARÍTIMO",
    287: "DIREITO PENAL",
    11068: "DIREITO PENAL MILITAR",
    195: "DIREITO PREVIDENCIÁRIO",
    8826: "DIREITO PROCESSUAL CIVIL E DO TRABALHO",
    1209: "DIREITO PROCESSUAL PENAL",
    11049: "DIREITO PROCESSUAL PENAL MILITAR",
    14: "DIREITO TRIBUTÁRIO",
    12467: "QUESTÕES DE ALTA COMPLEXIDADE, GRANDE IMPACTO E REPERCUSSÃO",
    7724: "REGISTROS PÚBLICOS",
}


# ================================================================
# 2. OBJETOS DE DADOS E LOGS
# ================================================================

@dataclass(frozen=True)
class Subject:
    """Representa um assunto e sua localização na árvore do SGT."""

    code: int
    description: str
    parent_code: int | None
    level: int
    status: str
    has_children: bool
    path: str


@dataclass(frozen=True)
class ChildSubject:
    """Representa a resposta mínima necessária para continuar a busca."""

    code: int
    description: str
    parent_code: int
    status: str
    has_children: bool


class SubjectGateway(Protocol):
    """Contrato isolado para permitir testes sem internet."""

    def list_children(self, parent_code: int) -> list[ChildSubject]:
        """Retorna os filhos diretos de um assunto."""
        ...

    def get_version(self) -> str:
        """Retorna a identificação da versão, quando disponível."""
        ...


class JsonLogFormatter(logging.Formatter):
    """Evita logs livres e facilita a auditoria de falhas de coleta."""

    def format(self, record: logging.LogRecord) -> str:
        return json.dumps(
            {
                "level": record.levelname,
                "event": record.getMessage(),
                "module": record.name,
            },
            ensure_ascii=False,
        )


def configure_logging() -> logging.Logger:
    """Mantém um único handler para não duplicar logs no notebook."""
    logger = logging.getLogger("cnj_assuntos")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(JsonLogFormatter())
        logger.addHandler(handler)
    logger.propagate = False
    return logger


LOGGER = configure_logging()


# ================================================================
# 3. INTEGRAÇÃO COM O WEBSERVICE OFICIAL DO CNJ
# ================================================================

SOAP_NS = "http://schemas.xmlsoap.org/soap/envelope/"
WSDL_NS = "http://schemas.xmlsoap.org/wsdl/"
SOAP_BINDING_NS = "http://schemas.xmlsoap.org/wsdl/soap/"


def explain_network_error(exc: Exception) -> str:
    """Classifica erros HTTPS e sugere uma intervenção compatível com segurança."""
    import requests

    detail = str(exc).lower()
    if isinstance(exc, requests.exceptions.SSLError):
        return ("Falha ao validar o certificado HTTPS. Em rede corporativa, "
                "solicite à TI o certificado raiz aprovado e configure REQUESTS_CA_BUNDLE. "
                "Não use verify=False.")
    if isinstance(exc, requests.exceptions.ProxyError):
        return ("Falha no proxy da rede. Confirme com a TI se o endereço do CNJ "
                "está liberado e se HTTPS_PROXY/HTTP_PROXY estão configurados corretamente.")
    if isinstance(exc, requests.exceptions.ConnectTimeout):
        return "Tempo excedido ao estabelecer conexão. Teste a URL no navegador ou use a exportação Excel oficial."
    if isinstance(exc, requests.exceptions.ReadTimeout):
        return "CNJ demorou a responder após conectar. Reduza os workers ou use os arquivos Excel oficiais."
    if isinstance(exc, requests.exceptions.HTTPError) and exc.response is not None:
        status = exc.response.status_code
        if status == 403:
            return "HTTP 403: acesso ao serviço recusado; use exportação Excel oficial ou procure suporte do CNJ."
        if status == 404:
            return "HTTP 404: rota não disponível; confira o endereço publicado pelo CNJ."
        if status == 429:
            return "HTTP 429: limite de requisições; reduza o paralelismo e respeite o intervalo recomendado."
        if status >= 500:
            return f"HTTP {status}: falha no servidor CNJ; tente novamente depois ou use a exportação Excel oficial."
        return f"HTTP {status}: resposta inesperada do serviço CNJ."
    if any(term in detail for term in ("nameresolutionerror", "name or service not known",
                                       "getaddrinfo", "temporary failure in name resolution", "gethostbyname")):
        return "Falha de DNS: o computador não resolveu www.cnj.jus.br. Confira DNS, VPN e acesso externo com a TI."
    if "connection refused" in detail or "actively refused" in detail:
        return "O servidor ou rede recusou a conexão HTTPS. Valide firewall, proxy e disponibilidade do serviço."
    if "connection reset" in detail or "remote end closed" in detail:
        return "Conexão encerrada por rede ou servidor. Diminua o paralelismo ou use a exportação Excel oficial."
    return ("Falha HTTPS sem causa identificada pela mensagem resumida. "
            "Consulte a última linha do traceback ('Caused by') e teste o modo diagnose.")


def raise_service_error(exc: Exception, url: str) -> None:
    """Acrescenta diagnóstico sem esconder a exceção original."""
    raise ConnectionError(f"Não foi possível consultar {url}. {explain_network_error(exc)}") from exc


def diagnose_connection(wsdl_url: str, timeout: int = 8) -> bool:
    """Executa um teste rápido da URL pública, sem coleta nem credenciais."""
    import requests
    print(f"Testando serviço do CNJ: {wsdl_url}")
    try:
        response = requests.get(wsdl_url, timeout=(timeout, timeout), headers={"User-Agent": "CNJ-Assuntos-Research/2.0"})
        response.raise_for_status()
        if b"definitions" not in response.content[:10000]:
            print("HTTP OK, mas resposta não parece um WSDL. Confira o acesso e o endereço.")
            return False
        print("HTTP OK: contrato SOAP disponível.")
        return True
    except requests.exceptions.RequestException as exc:
        print(f"Diagnóstico: {explain_network_error(exc)}")
        print(f"Tipo de erro: {type(exc).__name__}")
        return False


def get_field(value: Any, field: str) -> Any:
    """Aceita respostas de teste em objetos ou dicionários."""
    if isinstance(value, Mapping):
        return value.get(field)
    return getattr(value, field, None)


def parse_has_children(value: Any) -> bool:
    """Interpreta sem ambiguidades a indicação de descendentes."""
    if value is None or value == "":
        return False
    if isinstance(value, bool):
        return value
    normalized = str(value).strip().upper()
    if normalized in {"1", "S", "SIM", "TRUE", "Y"}:
        return True
    if normalized in {"0", "N", "NAO", "NÃO", "FALSE", "NO"}:
        return False
    raise ValueError(f"Valor inesperado no campo 'temFilhos': {value!r}")


def local_name(tag: str) -> str:
    """Remove o namespace sem alterar o conteúdo textual do XML."""
    return tag.split("}", 1)[-1].split(":")[-1]


def find_child(element: ET.Element, name: str) -> ET.Element | None:
    """Localiza o campo pelo nome, independentemente de prefixos SOAP."""
    return next((child for child in element if local_name(child.tag) == name), None)


def resolve_reference(element: ET.Element, references: Mapping[str, ET.Element]) -> ET.Element:
    """Segue referências SOAP 'href=#id', inclusive em múltiplos níveis."""
    observed: set[str] = set()
    while True:
        href = next((str(value) for key, value in element.attrib.items()
                     if local_name(key) == "href"), None)
        if not href:
            return element
        identifier = href.lstrip("#")
        if identifier in observed or identifier not in references:
            raise ValueError(f"Referência SOAP inválida ou circular: {identifier}")
        observed.add(identifier)
        element = references[identifier]


def parse_soap_document(payload: bytes, operation: str) -> ET.Element:
    """Valida o XML SOAP e recupera a resposta da operação solicitada."""
    root = ET.fromstring(payload)
    body = root.find(f"{{{SOAP_NS}}}Body")
    if body is None:
        raise ValueError("Resposta sem SOAP Body")

    fault = find_child(body, "Fault")
    if fault is not None:
        message = find_child(fault, "faultstring")
        raise RuntimeError(
            f"CNJ retornou SOAP Fault: {message.text if message is not None else 'sem mensagem'}"
        )

    response = find_child(body, f"{operation}Response")
    if response is None:
        raise ValueError(f"Resposta SOAP sem '{operation}Response'")
    return body


def parse_soap_return(payload: bytes, operation: str) -> ET.Element | None:
    """Identifica o retorno SOAP, resolvendo arrays com multiRef."""
    body = parse_soap_document(payload, operation)
    references = {
        identifier: item
        for item in body.iter()
        for key, identifier in item.attrib.items()
        if local_name(key) == "id"
    }
    response = find_child(body, f"{operation}Response")
    assert response is not None
    value = find_child(response, "return")
    return resolve_reference(value, references) if value is not None else None


def decode_child_response(payload: bytes) -> list[ChildSubject]:
    """Converte tanto SOAP direto quanto arrays PHP SOAP/multiRef."""
    operation = "getArrayFilhosItemPublicoWS"
    body = parse_soap_document(payload, operation)
    references = {
        str(identifier): item
        for item in body.iter()
        for key, identifier in item.attrib.items()
        if local_name(key) == "id"
    }
    response = find_child(body, f"{operation}Response")
    assert response is not None
    return_element = find_child(response, "return")
    if return_element is None:
        return []
    collection = resolve_reference(return_element, references)
    if any(local_name(key) == "nil" and str(value).lower() in ("true", "1")
           for key, value in collection.attrib.items()):
        return []

    entries = [resolve_reference(child, references) for child in collection
               if local_name(child.tag) in {"item", "multiRef"}]
    if not entries and find_child(collection, "seq_elemento") is not None:
        entries = [collection]
    if not entries and len(collection) > 0:
        raise ValueError("Resposta contém estrutura desconhecida, sem elementos 'item'")

    result: list[ChildSubject] = []
    for entry in entries:
        fields = {
            local_name(field.tag): (resolve_reference(field, references).text or "").strip()
            for field in entry
        }
        if not fields.get("seq_elemento") or not fields.get("dsc_elemento"):
            raise ValueError(f"Filho retornado com código ou descrição ausente: {fields}")
        if not fields.get("seq_elemento_pai"):
            raise ValueError(f"Filho retornado sem código pai: {fields['seq_elemento']}")
        result.append(
            ChildSubject(
                code=int(fields["seq_elemento"]),
                description=fields["dsc_elemento"],
                parent_code=int(fields["seq_elemento_pai"]),
                status=fields.get("situacao", ""),
                has_children=parse_has_children(fields.get("temFilhos")),
            )
        )
    return result


class CnjSgtGateway:
    """Cliente HTTP para o SOAP antigo do CNJ, sem depender de Zeep.

    O SGT utiliza serialização SOAP herdada de versões antigas do PHP. Ler
    diretamente seu XML evita falhas conhecidas em clientes SOAP genéricos.
    """

    def __init__(
        self,
        wsdl_url: str = WSDL_URL,
        timeout: int = NETWORK_TIMEOUT_SECONDS,
        retries: int = NETWORK_RETRIES,
    ) -> None:
        if not wsdl_url.startswith("https://") or "?wsdl" not in wsdl_url:
            raise ValueError("Informe a URL HTTPS oficial do WSDL, terminada em ?wsdl.")
        self.wsdl_url = wsdl_url
        self.endpoint = wsdl_url.partition("?")[0]
        self.timeout = timeout
        self.retries = retries
        self._thread_local = threading.local()
        self._meta: dict[str, tuple[tuple[str, ...], str]] = {}

        # Lê o contrato oficial uma única vez; os workers reutilizam seus metadados.
        try:
            response = self._session().get(self.wsdl_url, timeout=(10, self.timeout))
            response.raise_for_status()
        except Exception as exc:
            # A conexão pode falhar antes de qualquer código SOAP ser executado.
            raise_service_error(exc, self.wsdl_url)
        self._load_wsdl(response.content)

    def _session(self) -> Any:
        """Mantém conexões HTTP independentes e reutilizáveis em cada worker."""
        current = getattr(self._thread_local, "session", None)
        if current is not None:
            return current
        try:
            import requests
            from requests.adapters import HTTPAdapter
            from urllib3.util.retry import Retry
        except ImportError as exc:
            raise RuntimeError("Instale a biblioteca requests: pip install requests") from exc

        session = requests.Session()
        session.headers.update({"User-Agent": "CNJ-Assuntos-Research/1.0"})
        retry = Retry(
            total=self.retries,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=None,  # Somente operações SOAP de consulta (leitura).
            respect_retry_after_header=True,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=2, pool_maxsize=2)
        session.mount("https://", adapter)
        self._thread_local.session = session
        return session

    def _load_wsdl(self, payload: bytes) -> None:
        """Obtém nomes reais dos argumentos e SOAPAction do WSDL do CNJ."""
        root = ET.fromstring(payload)
        self.namespace = root.attrib.get("targetNamespace", "urn:sgt")
        for operation in ("getArrayFilhosItemPublicoWS", "getDataUltimaVersao"):
            request = next(
                (node for node in root.findall(f".//{{{WSDL_NS}}}message")
                 if node.attrib.get("name") == f"{operation}Request"),
                None,
            )
            if request is None:
                if operation == "getDataUltimaVersao":
                    continue
                raise ValueError(f"WSDL não declara a requisição '{operation}'")

            parts = tuple(part.attrib["name"] for part in request
                          if local_name(part.tag) == "part" and "name" in part.attrib)
            if operation == "getArrayFilhosItemPublicoWS" and len(parts) != 2:
                raise ValueError(f"Assinatura SOAP inesperada: {operation}: {parts}")
            action = f"{self.namespace}#{operation}"
            for binding in root.findall(f".//{{{WSDL_NS}}}binding"):
                for method in binding.findall(f"{{{WSDL_NS}}}operation"):
                    if method.attrib.get("name") == operation:
                        soap_info = method.find(f"{{{SOAP_BINDING_NS}}}operation")
                        if soap_info is not None:
                            action = soap_info.attrib.get("soapAction", action)
            self._meta[operation] = (parts, action)

    def _post(self, operation: str, arguments: tuple[str, ...]) -> bytes:
        """Realiza consulta SOAP conforme os nomes de parâmetros do contrato."""
        if operation not in self._meta:
            raise ValueError(f"Operação não declarada pelo WSDL: {operation}")
        parts, action = self._meta[operation]
        if len(parts) != len(arguments):
            raise ValueError(f"Argumentos incompatíveis para operação {operation}")

        envelope = ET.Element(f"{{{SOAP_NS}}}Envelope")
        body = ET.SubElement(envelope, f"{{{SOAP_NS}}}Body")
        call = ET.SubElement(body, f"{{{self.namespace}}}{operation}")
        for field_name, argument in zip(parts, arguments):
            ET.SubElement(call, f"{{{self.namespace}}}{field_name}").text = argument
        payload = ET.tostring(envelope, encoding="utf-8", xml_declaration=True)

        try:
            response = self._session().post(
                self.endpoint,
                data=payload,
                headers={"Content-Type": "text/xml; charset=utf-8", "SOAPAction": f'"{action}"'},
                timeout=(10, self.timeout),
            )
            response.raise_for_status()
        except Exception as exc:
            raise_service_error(exc, self.endpoint)
        return response.content

    def list_children(self, parent_code: int) -> list[ChildSubject]:
        """Consulta filhos de um código e verifica que os pais são coerentes."""
        payload = self._post("getArrayFilhosItemPublicoWS", (str(parent_code), "A"))
        children = decode_child_response(payload)
        for child in children:
            if child.parent_code != parent_code:
                raise ValueError(
                    f"Assunto {child.code}: pai informado {child.parent_code} "
                    f"diverge da consulta {parent_code}"
                )
        return children

    def get_version(self) -> str:
        """Consulta versão quando o método estiver publicado no WSDL."""
        if "getDataUltimaVersao" not in self._meta:
            return ""
        try:
            payload = self._post("getDataUltimaVersao", ())
            result = parse_soap_return(payload, "getDataUltimaVersao")
            return "" if result is None else "".join(result.itertext()).strip()
        except Exception as exc:
            LOGGER.warning("Versão do serviço não informada: %s", exc)
            return ""


# ================================================================
# 4. PERCURSO COMPLETO E VALIDAÇÃO DA HIERARQUIA
# ================================================================


class SubjectCollector:
    """Percorre somente nós que possuem filhos, com paralelismo controlado."""

    def __init__(
        self,
        gateway: SubjectGateway,
        roots: Mapping[int, str],
        workers: int = MAX_WORKERS,
    ) -> None:
        if not roots or workers < 1 or workers > 8:
            raise ValueError("Informe categorias raiz e entre 1 e 8 workers.")
        self.gateway = gateway
        self.roots = dict(roots)
        self.workers = workers

    def collect(self) -> dict[int, Subject]:
        """Retorna todos os nós conhecidos ou falha se algum ramo não for consultado."""
        found: dict[int, Subject] = {}
        pending: deque[Subject] = deque()

        # Todas as raízes são consultadas, inclusive quando seu status é desconhecido.
        for code, description in self.roots.items():
            if code <= 0 or not description.strip():
                raise ValueError(f"Categoria raiz inválida: {code}")
            root = Subject(code, description.strip(), None, 1, "", True, description.strip())
            found[code] = root
            pending.append(root)

        scanned = 0
        next_progress = 250
        LOGGER.info("Iniciando varredura de %d categorias de topo", len(pending))

        with ThreadPoolExecutor(max_workers=self.workers) as pool:
            while pending:
                # Limita simultaneidade e não gera milhares de requisições antecipadas.
                batch = [pending.popleft() for _ in range(min(self.workers, len(pending)))]
                futures = {pool.submit(self.gateway.list_children, node.code): node for node in batch}

                for future in as_completed(futures):
                    parent = futures[future]
                    try:
                        children = future.result()
                    except Exception as exc:
                        raise RuntimeError(
                            f"Não foi possível concluir o ramo {parent.code} - "
                            f"{parent.description}. CSV final não será gerado."
                        ) from exc
                    scanned += 1

                    if parent.has_children and parent.level > 1 and not children:
                        raise ValueError(
                            f"O servidor anunciou descendentes para {parent.code}, "
                            "mas não devolveu nenhum. Interrompendo para evitar CSV incompleto."
                        )

                    for child in children:
                        existing = found.get(child.code)
                        if existing is not None:
                            if existing.parent_code != child.parent_code:
                                raise ValueError(
                                    f"Código {child.code} apareceu em pais diferentes: "
                                    f"{existing.parent_code} e {child.parent_code}"
                                )
                            continue

                        subject = Subject(
                            code=child.code,
                            description=child.description,
                            parent_code=parent.code,
                            level=parent.level + 1,
                            status=child.status,
                            has_children=child.has_children,
                            path=f"{parent.path} > {child.description}",
                        )
                        found[subject.code] = subject
                        if subject.has_children:
                            pending.append(subject)

                if len(found) >= next_progress:
                    LOGGER.info(
                        "Andamento: %d assuntos; %d ramos consultados; %d ramos pendentes",
                        len(found), scanned, len(pending),
                    )
                    next_progress += 250

        LOGGER.info("Consulta concluída: %d assuntos, %d ramos", len(found), scanned)
        return found


# ================================================================
# 5. EXPORTAÇÃO ATÔMICA DE UM ÚNICO CSV
# ================================================================


def export_csv(subjects: Mapping[int, Subject], output: Path, version: str) -> None:
    """Grava CSV somente após validar a abrangência da extração."""
    if len(subjects) < MIN_EXPECTED_RECORDS:
        raise ValueError(
            f"Somente {len(subjects)} assuntos encontrados. "
            f"Esperados pelo menos {MIN_EXPECTED_RECORDS}: saída não gerada."
        )
    if max(subject.level for subject in subjects.values()) < 3:
        raise ValueError("A árvore parece truncada: não foram encontrados 3 níveis.")
    for code, subject in subjects.items():
        if not subject.description or (subject.parent_code is not None and subject.parent_code not in subjects):
            raise ValueError(f"Assunto inválido ou sem pai conhecido: {code}")

    output = Path(output).expanduser().resolve()
    output.parent.mkdir(parents=True, exist_ok=True)
    temp_path: Path | None = None

    try:
        # O tempfile na mesma pasta permite substituir o arquivo final atomicamente.
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8-sig", newline="", suffix=".csv.tmp",
            dir=output.parent, delete=False,
        ) as stream:
            temp_path = Path(stream.name)
            writer = csv.writer(stream, delimiter=";", quoting=csv.QUOTE_MINIMAL)
            writer.writerow(
                ["codigo", "descricao", "codigo_pai", "nivel", "situacao", "caminho", "versao_sgt"]
            )
            # Ordenação pelo caminho mantém grupos de assuntos próximos no Excel.
            for subject in sorted(subjects.values(), key=lambda row: (row.path.casefold(), row.code)):
                writer.writerow(
                    [subject.code, subject.description, subject.parent_code or "", subject.level,
                     subject.status, subject.path, version]
                )
        os.replace(temp_path, output)
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)

    digest = hashlib.sha256(output.read_bytes()).hexdigest()
    LOGGER.info("CSV gerado: %s; linhas=%d; sha256=%s", output, len(subjects), digest)


def main(argv: list[str] | None = None) -> None:
    """Permite executar online, diagnosticar HTTPS ou converter Excel oficial."""
    parser = argparse.ArgumentParser(description="Gera CSV dos assuntos processuais do CNJ")
    parser.add_argument("--mode", choices=("soap", "diagnose", "excel"), default="soap")
    parser.add_argument("--input-dir", type=Path, default=Path("planilhas_cnj"))
    parser.add_argument("--output", type=Path, default=OUTPUT_CSV)
    parser.add_argument("--wsdl", default=WSDL_URL)
    parser.add_argument("--workers", type=int, default=MAX_WORKERS)
    parser.add_argument("--timeout", type=int, default=NETWORK_TIMEOUT_SECONDS)
    parser.add_argument("--retries", type=int, default=NETWORK_RETRIES)
    args = parser.parse_args(argv)
    if args.timeout < 1 or args.retries < 0:
        parser.error("Informe timeout >= 1 e retries >= 0")

    if args.mode == "diagnose":
        diagnose_connection(args.wsdl, min(args.timeout, 10))
        return

    if args.mode == "excel":
        from offline_excel import convert_excel_directory
        convert_excel_directory(args.input_dir, args.output)
        return

    gateway = CnjSgtGateway(wsdl_url=args.wsdl, timeout=args.timeout, retries=args.retries)
    version = gateway.get_version()
    LOGGER.info("Versão informada pelo SGT: %s", version or "não disponível")
    collector = SubjectCollector(gateway=gateway, roots=ROOT_SUBJECTS, workers=args.workers)
    subjects = collector.collect()
    export_csv(subjects, args.output, version)
    print(f"\nConcluído: {len(subjects)} assuntos salvos em {args.output.resolve()}")


if __name__ == "__main__":
    main()

'''
exec(compile(source, "extrair_assuntos_cnj.py", "exec"), cnj_module.__dict__)
print("Extrator de assuntos carregado.")


Extrator de assuntos carregado.


In [4]:
# CONFIGURAÇÕES: altere somente os valores abaixo.
# "diagnose": verifica erros de rede sem iniciar a coleta.
# "excel": gera CSV a partir das planilhas oficiais em INPUT_DIR (sem HTTPS).
# "soap": percorre a árvore do CNJ por HTTPS quando o serviço estiver acessível.
from pathlib import Path

MODE = "diagnose"
INPUT_DIR = Path("planilhas_cnj")
OUTPUT_CSV = Path("cnj_assuntos_completos.csv")
WORKERS = 2
RETRIES = 2
TIMEOUT = 20

cnj_module.main([
    "--mode", MODE,
    "--input-dir", str(INPUT_DIR),
    "--output", str(OUTPUT_CSV),
    "--workers", str(WORKERS),
    "--retries", str(RETRIES),
    "--timeout", str(TIMEOUT),
])


c:\Users\caioa\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


Testando serviço do CNJ: https://www.cnj.jus.br/sgt/sgt_ws.php?wsdl
HTTP OK: contrato SOAP disponível.
